# OccamRoute — Surgical AST Context Slicing & Cost-Quality LLM Routing Benchmark (Colab GPU)

**Official Repository:** [maykonlincolnusa/OccamRoute-Surgical-Context-Complexity-Routing-Engine-](https://github.com/maykonlincolnusa/OccamRoute-Surgical-Context-Complexity-Routing-Engine-)

This 1-click GPU notebook benchmarks **3 competing context-retrieval and model-routing pipelines** side-by-side on NVIDIA **T4 / L4** GPUs:
1. **Baseline A — Brute Force:** Full-module source injection routed unconditionally to frontier models.
2. **Baseline B — Naive RAG (FAISS + `BAAI/bge-small-en-v1.5`):** Overlapping 512-token text chunking with cosine retrieval.
3. **OccamRoute (ML):** Deterministic **AST + PageRank k-hop Slicer**, **U-shaped Position-Aware Ordering** (*Lost in the Middle* mitigation), and calibrated **PyTorch MLP Router** offloading low-complexity queries to a live **4-bit NF4 `Qwen/Qwen2.5-Coder-7B-Instruct`** model.

In [ ]:
#@title 1. Verify Colab GPU & Install OccamRoute + GPU Inference Stack
import os, subprocess, sys, torch

assert torch.cuda.is_available(), "Enable Runtime -> Change runtime type -> T4 GPU or L4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

if not os.path.exists("/content/OccamRoute"):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/maykonlincolnusa/OccamRoute-Surgical-Context-Complexity-Routing-Engine-.git",
        "/content/OccamRoute"
    ], check=True)

%cd /content/OccamRoute
%pip install -q -e . faiss-cpu sentence-transformers transformers accelerate bitsandbytes datasets tiktoken matplotlib

In [ ]:
#@title 2. Ingest SWE-bench Lite & Build Real Repository AST Graph (`encode/httpx@0.28.1`)
import subprocess
from pathlib import Path
from occamroute import RepoGraphBuilder, load_swebench_lite

swe_sample = load_swebench_lite(limit=25, split="test")
print(f"Loaded {len(swe_sample)} real tasks from princeton-nlp/SWE-bench_Lite")

httpx_dir = Path("/content/httpx")
if not (httpx_dir / "httpx").exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "0.28.1",
        "https://github.com/encode/httpx.git", str(httpx_dir)
    ], check=True)

repo_graph = RepoGraphBuilder().build_from_directory(httpx_dir / "httpx")
print("AST RepoGraph Summary:", repo_graph.summary_stats())

In [ ]:
#@title 3. Load Trained GBDT Router & 4-bit NF4 Qwen2.5-Coder-7B-Instruct
import json, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from occamroute import OccamRouter

metrics_meta = json.loads(Path("artifacts/router_metrics.json").read_text())
optimal_tau = float(metrics_meta.get("optimal_tau", 0.30))
router = OccamRouter.from_sklearn_artifacts("artifacts", threshold=optimal_tau)
print(f"Loaded OccamRouter ({router.backend_name}) with calibrated threshold tau={optimal_tau:.2f}")

model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
local_llm = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
local_llm.eval()
print(f"4-bit Model Peak VRAM: {torch.cuda.max_memory_allocated(0) / (1024**3):.2f} GB")

def run_local_4bit(query: str, context: str) -> tuple[str, int, int, float]:
    msgs = [
        {"role": "system", "content": "Cite exact file paths and symbol names from the context."},
        {"role": "user", "content": f"Context:\n{context}\n\nTask:\n{query}"}
    ]
    prompt_txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_txt, return_tensors="pt", truncation=True, max_length=4096).to(local_llm.device)
    in_tok = int(inputs["input_ids"].shape[1])
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = local_llm.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000.0
    answer = tokenizer.decode(out[0][in_tok:], skip_special_tokens=True).strip()
    return answer, in_tok, int(out.shape[1] - in_tok), ms

In [ ]:
#@title 4. Run 3-Pipeline Comparative Benchmark & Print KPI Table
from occamroute.benchmark import (
    SWEBenchBenchmarkEvaluator,
    build_repo_eval_tasks,
    generate_publication_charts,
)

eval_tasks = build_repo_eval_tasks(repo_graph, max_tasks=12)
evaluator = SWEBenchBenchmarkEvaluator(
    repo_graph=repo_graph,
    router=router,
    rag_backend="auto",
    token_budget=2000,
    max_symbols=6,
    k_hop=2,
    top_k_rag=5,
    local_llm_fn=run_local_4bit,
)
report = evaluator.evaluate(eval_tasks)
saved_data = report.save("artifacts/colab_benchmark")
print(f"Q_weak={report.q_weak:.4f} | Q_strong={report.q_strong:.4f} | RAG backend={report.metadata['rag_backend']}")
display(report.summary_df.round(3))

In [ ]:
#@title 5. Generate & Display Dark-Theme Publication Charts (`docs/assets/`)
from IPython.display import Image, display

chart_paths = generate_publication_charts(report, output_dir="docs/assets")
print("Generated publication charts:", chart_paths)
display(Image(filename=str(chart_paths["pareto_frontier"])))
display(Image(filename=str(chart_paths["token_compression"])))